In [1]:
import os
BASE_DIR = "\\notebooks\\experiments"
if os.getcwd().endswith(BASE_DIR):
    os.chdir(os.path.abspath(os.path.join(os.getcwd(), '..', '..')))

In [2]:
from src.models.model_handler import save_model, load_model, get_class_def_from_notebook
from src.models.testing_utils import evaluate_continuous, predict_continuous
from src.data.preprocessing import generate_classification_tensors
from src.data.DatasetGenerator import DatasetGenerator
from src.models.training_utils import train_regressor_model
from sklearn.preprocessing import LabelEncoder
from src.utils.helpers import load_param

from torch.utils.data import TensorDataset, DataLoader
import torch.nn.functional as F
import torch.nn as nn
import pandas as pd
import numpy as np
import torch

In [3]:
import pandas as pd

num_particules = 10000

dg = DatasetGenerator(num_particules)

In [ ]:
import pickle
with open('./data/generated/all_types_200p_70000t_rp.pkl', 'rb') as file:
    data = pickle.load(file)

data = data.sample(frac=1).reset_index(drop=True)
data

,coords,trajectory_type
0,"[[1.7234899456325379, 2.5340802619738865, 0.55...",ATTM
1,"[[0.5940272587357962, -2.5906601636655653, -2....",ATTM
2,"[[1.3020598393048155, -2.3389714864390454, -2....",ATTM
1,"[[-0.5684039073565686, 0.5390483539845995, 0.9...",Lévy Walk
2,"[[0.0, -0.05076348206937256, -0.05490309225672...",Fractional Brownian Motion
3,"[[2.8209162597843784, 1.1614508785145126, -0.9...",ATTM
4,"[[-0.9957704562537888, -2.5277994770773615, -2...",ATTM
...,...,...
69995,"[[0.0, 2.241513562623141, 8.512472120425926, 8...",Brownian Motion
69996,"[[0.0, -4.102006445565995, -0.8982351543806684...",Brownian Motion


In [7]:
print(data.columns)

Index(['coords', 'trajectory_type'], dtype='object')


In [8]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.preprocessing import LabelEncoder
from torch.utils.data import DataLoader, TensorDataset

In [9]:
valores_unicos = data['trajectory_type'].unique()
print(valores_unicos)

['ATTM' 'CTRW' 'Fractional Brownian Motion' 'Lévy Walk' 'Random Walk'
 'Scaled Brownian Motion' 'Brownian Motion']


In [10]:
# 1. Convertir las coordenadas a tensor 3D
# Asumiendo que 'data' es tu DataFrame
coords_list = [np.stack([row[0], row[1]], axis=1) for row in data['coords']]
X = np.array(coords_list)  # Forma (n_samples, seq_length, 2)
X_tensor = torch.tensor(X, dtype=torch.float32)

In [11]:
# 2. Convertir las etiquetas a numéricas
le = LabelEncoder()
labels = le.fit_transform(data['trajectory_type'])
labels_tensor  = torch.tensor(labels, dtype=torch.long)
y_tensor = F.one_hot(labels_tensor, num_classes=7).float()

In [12]:
# 3. Reorganizar dimensiones para Conv1D (n_samples, channels, seq_length)
X_tensor = X_tensor.permute(0, 2, 1)  # Nueva forma: (n, 2, m)

In [13]:
# 4. Dividir datos en entrenamiento y prueba (ejemplo simple)
train_size = int(0.8 * len(X_tensor))
X_train, X_test = X_tensor[:train_size], X_tensor[train_size:]
y_train, y_test = y_tensor[:train_size], y_tensor[train_size:]

In [15]:
print(y_tensor)
print(X_tensor)

tensor([[1., 0., 0.,  ..., 0., 0., 0.],
        [1., 0., 0.,  ..., 0., 0., 0.],
        [1., 0., 0.,  ..., 0., 0., 0.],
        ...,
        [0., 1., 0.,  ..., 0., 0., 0.],
        [0., 1., 0.,  ..., 0., 0., 0.],
        [0., 1., 0.,  ..., 0., 0., 0.]])
tensor([[[  1.7235,   2.5341,   0.5500,  ...,   1.6288,   3.6874,   3.0858],
         [ -0.3863,   0.8201,   0.6419,  ..., -20.7417, -18.7014, -19.8196]],

        [[  0.5940,  -2.5907,  -2.8704,  ...,  25.9566,  27.4888,  27.6969],
         [ -0.2470,   0.8017,  -0.1832,  ...,  -5.5172,  -5.6929,  -3.8071]],

        [[  1.3021,  -2.3390,  -2.7868,  ..., -59.2027, -59.5465, -58.3495],
         [  1.7126,  -0.3460,  -0.8940,  ...,   1.3309,   1.1350,   4.5362]],

        ...,

        [[  0.0000,  -0.7160,  -0.5491,  ...,  -9.1615,  -9.1705, -10.8355],
         [  0.0000,   0.5167,   2.2427,  ...,  17.4462,  16.4937,  16.9293]],

        [[  0.0000,  -1.1174,   1.1275,  ...,  94.7020,  88.0430,  83.6791],
         [  0.0000,  -1.8630,  

In [16]:
# 5. Crear DataLoader
batch_size = 32
train_dataset = TensorDataset(X_train, y_train)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

In [17]:
# 6. Definir la arquitectura CNN
class TrajectoryCNN(nn.Module):
    def __init__(self, input_channels, seq_length, num_classes):
        super().__init__()
        self.first_conv = nn.Conv1d(input_channels, 32, kernel_size=3, padding=1)
        self.conv1_32 = nn.Conv1d(32, 32, kernel_size=3, padding=1)
        self.conv2_32 = nn.Conv1d(32, 32, kernel_size=3, padding=1)
        self.conv_32_64 = nn.Conv1d(32, 64, kernel_size=3, padding=1)
        self.conv1_64 = nn.Conv1d(64, 64, kernel_size=3, padding=1)
        self.conv2_64 = nn.Conv1d(64, 64, kernel_size=3, padding=1)
        self.conv_64_128 = nn.Conv1d(64, 128, kernel_size=3, padding=1)
        self.conv1_128 = nn.Conv1d(128, 128, kernel_size=3, padding=1)
        self.conv2_128 = nn.Conv1d(128, 128, kernel_size=3, padding=1)
        self.max_pool = nn.MaxPool1d(2)
        self.avg_pool = nn.AdaptiveAvgPool2d((1, 4))
        self.fc1 = nn.Linear(4, 256)
        self.fc2 = nn.Linear(256, num_classes)

    
    def forward(self, data):
        loop = 2
        a_resnet = self.first_conv(data)
        a = self.conv1_32(a_resnet)
        a = self.conv2_32(a)
        a = a + a_resnet
        a = self.max_pool(a)
        b_resnet = self.conv_32_64(a)
        b = self.conv1_64(b_resnet)
        b = self.conv2_64(b)
        b = b + b_resnet
        b = self.max_pool(b)
        c_resnet = self.conv_64_128(b)
        c = self.conv1_128(c_resnet)
        c = self.conv2_128(c)
        c = c + c_resnet
        c = self.avg_pool(c)
        c = c.view(c.size(0), -1)  # Aplanar la salida
        c = F.relu(self.fc1(c))
        c = self.fc2(c)
        return c

In [18]:
# 7. Inicializar el modelo
model = TrajectoryCNN(
    input_channels=2,
    seq_length=X_tensor.shape[2],  # m (longitud de la secuencia)
    num_classes=len(le.classes_)
)

In [19]:
model.first_conv.weight.shape

torch.Size([32, 2, 3])

In [20]:
# 8. Configurar entrenamiento
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

In [21]:
# 9. Entrenar el modelo
num_epochs = 10

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    
    for inputs, labels in train_loader:
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
    
    print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {running_loss/len(train_loader):.4f}")


Epoch [1/10], Loss: 1.3523
Epoch [2/10], Loss: 0.7382
Epoch [3/10], Loss: 0.6002
Epoch [4/10], Loss: 0.5629
Epoch [5/10], Loss: 0.5352
Epoch [6/10], Loss: 0.5045
Epoch [7/10], Loss: 0.4687
Epoch [8/10], Loss: 0.4408
Epoch [9/10], Loss: 0.4048
Epoch [10/10], Loss: 0.3826


In [22]:
# 10. Evaluación (opcional)
with torch.no_grad():
    test_outputs = model(X_test)
    _, predicted = torch.max(test_outputs, 1)  # Índices de las clases predichas
    true_labels = torch.argmax(y_test, dim=1)  # Convertir one-hot a índices
    accuracy = (predicted == true_labels).float().mean()
    print(f'Test Accuracy: {accuracy.item():.4f}')

Test Accuracy: 0.8779


In [ ]:
import torch
import dill
import inspect

# Guardar el model fora d'un notebook
# inspect.getsource(TrajectoryCNN)

# Funció necessària per poder uardar el model dins d'un notebook
def get_class_def_from_notebook(class_name: str):
    from IPython import get_ipython
    shell = get_ipython()
    for cell in shell.history_manager.get_range():
        if class_name in cell[2]:
            return cell[2]

get_class_def_from_notebook("TrajectoryCNN")


'# 6. Definir la arquitectura CNN\nclass TrajectoryCNN(nn.Module):\n    def __init__(self, input_channels, seq_length, num_classes):\n        super().__init__()\n        self.first_conv = nn.Conv1d(input_channels, 32, kernel_size=3, padding=1)\n        self.conv1_32 = nn.Conv1d(32, 32, kernel_size=3, padding=1)\n        self.conv2_32 = nn.Conv1d(32, 32, kernel_size=3, padding=1)\n        self.conv_32_64 = nn.Conv1d(32, 64, kernel_size=3, padding=1)\n        self.conv1_64 = nn.Conv1d(64, 64, kernel_size=3, padding=1)\n        self.conv2_64 = nn.Conv1d(64, 64, kernel_size=3, padding=1)\n        self.conv_64_128 = nn.Conv1d(64, 128, kernel_size=3, padding=1)\n        self.conv1_128 = nn.Conv1d(128, 128, kernel_size=3, padding=1)\n        self.conv2_128 = nn.Conv1d(128, 128, kernel_size=3, padding=1)\n        self.max_pool = nn.MaxPool1d(2)\n        self.avg_pool = nn.AdaptiveAvgPool2d((1, 4))\n        self.fc1 = nn.Linear(4, 256)\n        self.fc2 = nn.Linear(256, num_classes)\n\n    \n  

In [ ]:
import dill

def save_model_with_source(model_instance, filename="model.pth"):
    torch.save({
        "model_name": model_instance.__class__.__name__,
        "model_type": "type_classifier",
        "model_source": get_class_def_from_notebook(model_instance.__class__.__name__), # Guardar el model fora d'un notebook: inspect.getsource(model_instance.__class__),
        "state_dict": model_instance.state_dict(),
        "attributes": {
            "input_channels": 2,
            "seq_length": X_tensor.shape[2],
            "num_classes": len(le.classes_)
        },
        "input_channels": 2
    }, filename, pickle_module=dill)

save_model_with_source(model, filename="model.pth")

In [42]:
import torch
import dill

def load_dynamic_model(model_path):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    checkpoint = torch.load(model_path, map_location=device, pickle_module=dill)
    
    # Ejecutar el código fuente para recrear la clase
    exec(checkpoint["model_source"], globals())  # La clase se define en el scope global
    
    # Instanciar el modelo con sus argumentos originales
    model_class = globals()[checkpoint["model_source"].split("class ")[1].split("(")[0].strip()]
    model = model_class(**checkpoint["attributes"])
    model.load_state_dict(checkpoint["state_dict"])
    model.to(device)
    model.eval()
    return model

# Uso
model = load_dynamic_model("model.pth")
print(model)  # Muestra la estructura del modelo cargado

TrajectoryCNN(
  (first_conv): Conv1d(2, 32, kernel_size=(3,), stride=(1,), padding=(1,))
  (conv1_32): Conv1d(32, 32, kernel_size=(3,), stride=(1,), padding=(1,))
  (conv2_32): Conv1d(32, 32, kernel_size=(3,), stride=(1,), padding=(1,))
  (conv_32_64): Conv1d(32, 64, kernel_size=(3,), stride=(1,), padding=(1,))
  (conv1_64): Conv1d(64, 64, kernel_size=(3,), stride=(1,), padding=(1,))
  (conv2_64): Conv1d(64, 64, kernel_size=(3,), stride=(1,), padding=(1,))
  (conv_64_128): Conv1d(64, 128, kernel_size=(3,), stride=(1,), padding=(1,))
  (conv1_128): Conv1d(128, 128, kernel_size=(3,), stride=(1,), padding=(1,))
  (conv2_128): Conv1d(128, 128, kernel_size=(3,), stride=(1,), padding=(1,))
  (max_pool): MaxPool1d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (avg_pool): AdaptiveAvgPool2d(output_size=(1, 4))
  (fc1): Linear(in_features=4, out_features=256, bias=True)
  (fc2): Linear(in_features=256, out_features=7, bias=True)
)


In [46]:
# Seleccionar el primer ejemplo de X_test y añadir una dimensión batch
first_example = X_test[0].unsqueeze(0)  # shape: (1, 2, 200)

# Pasar por el modelo
model.eval()
with torch.no_grad():
    output = model(first_example)
    predicted_class = torch.argmax(output, dim=1).item()

# Mostrar el resultado
print("Predicción (índice de clase):", predicted_class)
print("Clase predicha:", le.inverse_transform([predicted_class])[0])
# Mostrar la clase real
true_class = torch.argmax(y_test[0]).item()
print("Clase real:", le.inverse_transform([true_class])[0])

Predicción (índice de clase): 5
Clase predicha: Random Walk
Clase real: Random Walk


In [49]:
model.__class__.__name__

'TrajectoryCNN'

In [27]:
# Guardar solo los parámetros del modelo
torch.save(model.state_dict(), 'trajectory_cnn_model_2.pth')

# Si deseas guardar el modelo completo
torch.save(model, 'trajectory_cnn_model_complete_2.pth')

In [ ]:
# Cargar solo los parámetros del modelo
model = TrajectoryCNN(
    input_channels=2,
    seq_length=X_tensor.shape[2],  # m (longitud de la secuencia)
    num_classes=len(le.classes_)
)
model.load_state_dict(torch.load('trajectory_cnn_model_2.pth'))
model.eval()

# Si deseas cargar el modelo completo
# model = torch.load('trajectory_cnn_model_complete.pth')
# model.eval()